# Rugby Rulebook Tree Structural Complexity Analysis

This notebook measures structural complexity of the hierarchical rulebook
trees (built by the tree-extraction notebook) and relates that complexity
to the growth of the game (cumulative test matches) and to rulebook length
(word count). For each year it computes, from the saved YAML trees:

- **Depth** — average and maximum path length from the tree's `center` node
- **Branching factor** — average number of children per non-leaf node
- **Node count** and **leaf count**

It does this for two tree families, both produced by the tree-extraction
notebook:
1. `hierarchical_trees/flattened/` — referred to below as the **flat**
   family (fewer collapsed heading levels)
2. `hierarchical_trees/original/` — referred to below as the **deep**
   family (every heading level as its own tree depth)

...then fits power-law curves (node/leaf count vs. cumulative matches, and
vs. word count) and plots everything.

## Requirements

```bash
pip install networkx pyyaml matplotlib numpy pandas scipy scikit-learn nltk
```

## One-time setup: NLTK data

`nltk.word_tokenize` (used later to compute word counts) needs data that
isn't bundled with the library.

## Imports

In [ ]:
import networkx as nx
from pathlib import Path 
import yaml
import matplotlib.pyplot as plt
from networkx.drawing.nx_agraph import graphviz_layout
import re
import string
import nltk
import numpy as np
import pandas as pd
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib as mpl
import scipy.optimize as opt
from sklearn.metrics import r2_score
import sys
import csv
import math

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')

## Configuration

This notebook expects a `data/` folder (next to the notebook) laid out as:

```
data/
├── rugby_test_matches_per_year_backfilled.csv
├── rugby_laws_per_year.csv
├── hierarchical_trees/
│   ├── final_scrum_branches/
│   |   ├── clean_1845tree.yaml/
│   |   ├── ...
│   |   └── clean_2026tree.yaml/
│   └── original/                  
│   |   ├── clean_1845tree.yaml/
│   |   ├── ...
│   |   └── clean_2026tree.yaml/
├── interdependencies/
│   ├── laws/                  (generated)
│   └── laws_plan_defs/        (generated)  
└── text_files/
    └── cleaned_relavent_texts/
        ├── 1845/
        ├── 1874-1889/
        ├── 1894-1924/
        ├── 1929/
        ├── 1934-1949/
        ├── 1954-1959/
        ├── 1969-1974/
        ├── 1985-1995/
        ├── 1997-1998/
        ├── 2000-2002/
        ├── 2003-2016/
        └── 2018-2026/
```

Update `DATA_DIR` if your data lives somewhere other than next to this
notebook.

In [ ]:
DATA_DIR = Path('data')

MATCHES_CSV_PATH = DATA_DIR / 'rugby_test_matches_per_year_backfilled.csv'
TEXTS_DIR = DATA_DIR / 'text_files' / 'cleaned_relavent_texts'
TREES_BASE = DATA_DIR / 'hierarchical_trees'
TREES_DIR = TREES_BASE / 'original'
TREES_DIR_FLAT = TREES_BASE / 'flattened'

## Cumulative test matches per year

Same as in the earlier text-visualization notebook: loads the CSV of
cumulative international test matches, by year, into a
`year -> total matches` dictionary.

In [ ]:
csv_path = MATCHES_CSV_PATH

years_matches = {}

with open(csv_path, 'r') as csvfile: # adds element to dictionary of year: total matches
    reader = csv.DictReader(csvfile)
    for row in reader:
        years_matches[int(row['year'])] = int(row['n_matches_sum'])

## Tree-building helper

Rebuilds a `networkx.DiGraph` from a saved YAML tree (up to 5 levels deep),
same structure as in the tree-extraction notebook but using a **directed**
graph here (`nx.DiGraph`) since branching factor is computed from
out-degree.

In [ ]:
def create_heirarchical_tree(data):
    G = nx.DiGraph()
    G.add_node('center', color='#006699', node_size=300)
    level1_no = 0

    # adds level1 nodes, connecting to center
    for edge in data['(full ruleset)']:
        temp_data1 = data['(full ruleset)']

        level1_node = next(iter(edge))
        level1_node_name = str(level1_node + ' ')
        level1_no += 1
        level2_no = 0

        G.add_node(level1_node_name, color='#336699', node_size=200)
        G.add_edge('center', level1_node_name)

        #print("LAW:", law_node_name)


        # adds level2 nodes, connecting to level1
        for edge in temp_data1[level1_no-1][level1_node]:
            temp_data2 = temp_data1[level1_no-1][level1_node]

            level2_node = next(iter(edge))
            level2_node_name = str(level1_node_name + level2_node + ' ')
            level2_no += 1
            level3_no = 0

            G.add_node(level2_node_name, color='#666699', node_size=160)
            G.add_edge(level1_node_name, level2_node_name)

            #print("HEADING:", heading_node_name)


            # adds level3, connecting to level2
            for edge in temp_data2[level2_no-1][level2_node]:
                temp_data3 = temp_data2[level2_no-1][level2_node]

                level3_node = next(iter(edge))
                level3_node_name = str(level2_node_name + level3_node + ' ')
                level3_no += 1
                level4_no = 0

                G.add_node(level3_node_name, color='#996699', node_size=120)
                G.add_edge(level2_node_name, level3_node_name)

                #print("SUBLAW:", sublaw_node_name)


                # adds level4, connecting to level3
                for edge in temp_data3[level3_no-1][level3_node]:

                    level4_node = next(iter(edge))
                    level4_node_name = str(level3_node_name + level4_node+ ' ')
                    level4_no += 1
                    level5_no = 0

                    G.add_node(level4_node_name, color='#CC6699', node_size=80)
                    G.add_edge(level3_node_name, level4_node_name)

                    #print("LOWER:", subsublaw_node_name)

                    # adds level5, connecting to level4
                    for edge in temp_data3[level3_no-1][level3_node][level4_no-1][level4_node]:
                        level5_node = next(iter(edge))
                        level5_node_name = str(level4_node_name + level5_node + ' ')
                        level5_no += 1
                        level6_no = 0

                        G.add_node(level5_node_name, color='#FF6699', node_size=60)
                        G.add_edge(level4_node_name, level5_node_name)

                        #print("ROMAN:", subsubsublaw_node_name)

                        # adds level6, connecting to level5
                        for edge in temp_data3[level3_no-1][level3_node][level4_no-1][level4_node][level5_no-1][level5_node]:
                            level6_node = next(iter(edge))
                            level6_node_name = str(level5_node_name + level6_node + ' ')
                            level6_no += 1
                            level7_no = 0

                            G.add_node(level6_node_name, color='#FF9999', node_size=40)
                            G.add_edge(level5_node_name, level6_node_name)


                            # adds level7, connecting to level6
                            for edge in temp_data3[level3_no-1][level3_node][level4_no-1][level4_node][level5_no-1][level5_node][level6_no-1][level6_node]:
                                level7_node = next(iter(edge))
                                level7_node_name = str(level6_node_name + level7_node + ' ')
                                level7_no += 1
                                level8_no = 0

                                G.add_node(level7_node_name, color='#FFCC66', node_size=35)
                                G.add_edge(level6_node_name, level7_node_name)

                                # adds level8, connecting to level7
                                for edge in temp_data3[level3_no-1][level3_node][level4_no-1][level4_node][level5_no-1][level5_node][level6_no-1][level6_node][level7_no-1][level7_node]:
                                    level8_node = next(iter(edge))
                                    level8_node_name = str(level7_node_name + level8_node + ' ')
                                    level8_no += 1
                                    level9_no = 0

                                    G.add_node(level8_node_name, color='#FFFF66', node_size=30)
                                    G.add_edge(level7_node_name, level8_node_name)

                                    # adds level9, connecting to level8
                                    for edge in temp_data3[level3_no-1][level3_node][level4_no-1][level4_node][level5_no-1][level5_node][level6_no-1][level6_node][level7_no-1][level7_node][level8_no-1][level8_node]:
                                        level9_node = str(edge)
                                        level9_node_name = str(level8_node_name + level9_node + ' ')
                                        level9_no += 1
                                        level10_no = 0

                                        G.add_node(level9_node_name, color='#CCFF66', node_size=25)
                                        G.add_edge(level8_node_name, level9_node_name)



    return G
        


## Load the flat trees

For every YAML in `TREES_DIR_FLAT`, rebuilds the graph and computes, per
year: path lengths from `center`, average branching factor (mean out-degree
over non-leaf nodes), node count, and leaf count.

In [ ]:
flat_folder_path = TREES_DIR_FLAT

lengths_dict = {}
branching_dict = {}
node_count_dict = {}
leaves_count_dict = {}

for file_path in (flat_folder_path.glob('*.yaml')):
    with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
        file_name = file_path.stem
        year = file_name.replace('clean_', '').replace('tree', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)
        
        lengths = nx.single_source_shortest_path_length(G, 'center')
        lengths_dict[int(year)] = lengths

        avg_branch= sum(branches for node, branches in G.out_degree if branches!=0) / len([node for node, branches in G.out_degree if branches!=0])
        branching_dict[int(year)] = avg_branch

        node_count_dict[int(year)] = len(G)
        leaves_count_dict[int(year)] = len([node for node, branches in G.out_degree if branches==0])



Same computation, again on the flat trees — kept as a separate pass/variable set in the original code.

In [ ]:
flat_p_folder_path = TREES_DIR_FLAT

lengths_dict_p = {}
branching_dict_p = {}
node_count_dict_p = {}
leaves_count_dict_p = {}

for file_path in (flat_p_folder_path.glob('*.yaml')):
    with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
        file_name = file_path.stem
        year = file_name.replace('clean_', '').replace('tree', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)
        
        lengths = nx.single_source_shortest_path_length(G, 'center')
        lengths_dict_p[int(year)] = lengths

        avg_branch= sum(branches for node, branches in G.out_degree if branches!=0) / len([node for node, branches in G.out_degree if branches!=0])
        branching_dict_p[int(year)] = avg_branch

        node_count_dict_p[int(year)] = len(G)
        leaves_count_dict_p[int(year)] = len([node for node, branches in G.out_degree if branches==0])



Same again, but for the **deep** (un-flattened) trees in `TREES_DIR` —
only path lengths are computed for this family.

In [ ]:
deep_folder_path = TREES_DIR

d_lengths_dict = {}

for file_path in (deep_folder_path.glob('*.yaml')):
    with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
        file_name = file_path.stem
        year = file_name.replace('clean_', '').replace('tree', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)
        
        lengths = nx.single_source_shortest_path_length(G, 'center')
        d_lengths_dict[int(year)] = lengths


## Derive matches-keyed versions

For each pass above, computes average/max depth (and, where available,
branching/node/leaf counts) keyed by year, and a second copy keyed by
cumulative matches (for years from 1871 onward, when match records begin).

In [ ]:
avg_lengths = {}
max_lengths = {}
avg_lengths_match = {}
max_lengths_match = {}
avg_branching_match = {}
node_count_match = {}
leaves_count_match = {}

# creating dictionary year: tokenized words
for year in lengths_dict:

    avg_lengths[year] = sum(lengths_dict[year].values()) / len(lengths_dict[year].values())
    max_lengths[year] = max(lengths_dict[year].values()) 

    if year>=1871:
        matches = years_matches[year]
        avg_lengths_match[matches] = sum(lengths_dict[year].values()) / len(lengths_dict[year].values())
        max_lengths_match[matches] = max(lengths_dict[year].values())
        avg_branching_match[matches] = branching_dict[year]
        node_count_match[matches] = node_count_dict[year]
        leaves_count_match[matches] = leaves_count_dict[year]


Same derivation for the second flat pass.

In [ ]:
avg_lengths_p = {}
max_lengths_p = {}
avg_lengths_match_p = {}
max_lengths_match_p = {}
avg_branching_match_p = {}
node_count_match_p = {}
leaves_count_match_p = {}

# creating dictionary year: tokenized words
for year in lengths_dict_p:

    avg_lengths_p[year] = sum(lengths_dict_p[year].values()) / len(lengths_dict_p[year].values())
    max_lengths_p[year] = max(lengths_dict_p[year].values()) 

    if year>=1871:
        matches = years_matches[year]
        avg_lengths_match_p[matches] = sum(lengths_dict_p[year].values()) / len(lengths_dict_p[year].values())
        max_lengths_match_p[matches] = max(lengths_dict_p[year].values())
        avg_branching_match_p[matches] = branching_dict_p[year]
        node_count_match_p[matches] = node_count_dict_p[year]
        leaves_count_match_p[matches] = leaves_count_dict_p[year]


Same derivation for the deep family (depth only).

In [ ]:
d_avg_lengths = {}
d_max_lengths = {}
d_avg_lengths_match = {}
d_max_lengths_match = {}

# creating dictionary year: tokenized words
for year in d_lengths_dict:

    d_avg_lengths[year] = sum(d_lengths_dict[year].values()) / len(d_lengths_dict[year].values())
    d_max_lengths[year] = max(d_lengths_dict[year].values()) 

    if year>=1871:
        matches = years_matches[year]
        d_avg_lengths_match[matches] = sum(d_lengths_dict[year].values()) / len(d_lengths_dict[year].values())
        d_max_lengths_match[matches] = max(d_lengths_dict[year].values())


## Average tree depth vs. year and vs. cumulative matches — first flat pass.

In [ ]:
# avg length by year
plt.figure(figsize=(5,5))
#plt.scatter(d_avg_lengths.keys(), d_avg_lengths.values(), color='turquoise')
plt.scatter(avg_lengths.keys(), avg_lengths.values(), color='black')
plt.ylabel('Average depth', size=16)
plt.xlabel('Year', size=16)

# avg length by matches
plt.figure(figsize=(5,5))
#plt.scatter(d_avg_lengths_match.keys(), d_avg_lengths_match.values(), color='turquoise')
plt.scatter(avg_lengths_match.keys(), avg_lengths_match.values(), color='black')
plt.xscale('log')
plt.ylabel('Average depth', size=16)
plt.xlabel('Cumulative matches', size=16)


Same plots for the second flat pass.

In [ ]:
# avg length by year
plt.figure(figsize=(5,5))
#plt.scatter(d_avg_lengths.keys(), d_avg_lengths.values(), color='turquoise')
plt.scatter(avg_lengths_p.keys(), avg_lengths_p.values(), color='black')
plt.ylabel('Average depth', size=16)
plt.xlabel('Year', size=16)

# avg length by matches
plt.figure(figsize=(5,5))
#plt.scatter(d_avg_lengths_match.keys(), d_avg_lengths_match.values(), color='turquoise')
plt.scatter(avg_lengths_match_p.keys(), avg_lengths_match_p.values(), color='black')
plt.xscale('log')
plt.ylabel('Average depth', size=16)
plt.xlabel('Cumulative matches', size=16)


## Maximum tree depth vs. year and vs. cumulative matches — first flat pass.

In [ ]:
# max length by year
plt.figure(figsize=(5,5))
#plt.scatter(d_max_lengths.keys(), d_max_lengths.values(), color='turquoise')
plt.scatter(max_lengths.keys(), max_lengths.values(), color='black')
plt.ylabel('Maximum depth', size=16)
plt.xlabel('Year', size=16)

# max length by matches
plt.figure(figsize=(5,5))
#plt.scatter(d_max_lengths_match.keys(), d_max_lengths_match.values(), color='turquoise')
plt.scatter(max_lengths_match.keys(), max_lengths_match.values(), color='black')
plt.xscale('log')
plt.ylabel('Maximum depth', size=16)
plt.xlabel('Cumulative matches', size=16)

Same plots for the second flat pass.

In [ ]:
# max length by year
plt.figure(figsize=(5,5))
#plt.scatter(d_max_lengths.keys(), d_max_lengths.values(), color='turquoise')
plt.scatter(max_lengths_p.keys(), max_lengths_p.values(), color='black')
plt.ylabel('Maximum depth', size=16)
plt.xlabel('Year', size=16)

# max length by matches
plt.figure(figsize=(5,5))
#plt.scatter(d_max_lengths_match.keys(), d_max_lengths_match.values(), color='turquoise')
plt.scatter(max_lengths_match_p.keys(), max_lengths_match_p.values(), color='black')
plt.xscale('log')
plt.ylabel('Maximum depth', size=16)
plt.xlabel('Cumulative matches', size=16)

## Branching factor vs. year and vs. cumulative matches — first flat pass.

In [ ]:
# branches by year
plt.figure(figsize=(5,5))
plt.scatter(branching_dict.keys(), branching_dict.values(), color='black')
plt.ylabel('Branches', size=16)
plt.xlabel('Year', size=16)

# branches by matches
plt.figure(figsize=(5,5))
plt.scatter(avg_branching_match.keys(), avg_branching_match.values(), color='black')
plt.xscale('log')
plt.ylabel('Branches', size=16)
plt.xlabel('Cumulative matches', size=16)

Same plots for the second flat pass.

In [ ]:
# branches by year
plt.figure(figsize=(5,5))
plt.scatter(branching_dict_p.keys(), branching_dict_p.values(), color='black')
plt.ylabel('Branches', size=16)
plt.xlabel('Year', size=16)

# branches by matches
plt.figure(figsize=(5,5))
plt.scatter(avg_branching_match_p.keys(), avg_branching_match_p.values(), color='black')
plt.xscale('log')
plt.ylabel('Branches', size=16)
plt.xlabel('Cumulative matches', size=16)

## Node/leaf counts vs. cumulative matches

Power-law model (`y = a * x^b`) fit to node count and leaf count vs.
cumulative matches. Sorts and converts to fittable arrays — first flat
pass.

In [ ]:
def count_match_func(x, a, b):
    y = a * (x ** b) 
    return y
initial_guesses = [0, 0]

sorted_node_counts = {matches: counts for matches, counts in sorted(node_count_match.items())}
matches_fittable = np.array(list(sorted_node_counts.keys()), dtype=float) # converts to fittable form
node_counts_fittable = np.array(list(sorted_node_counts.values()), dtype=float) 

sorted_leaves_counts = {matches: counts for matches, counts in sorted(leaves_count_match.items())}
matches_leaves_fittable = np.array(list(sorted_leaves_counts.keys()), dtype=float) # converts to fittable form
leaves_counts_fittable = np.array(list(sorted_leaves_counts.values()), dtype=float) 


Same, for the second flat pass.

In [ ]:
sorted_node_counts_p = {matches: counts for matches, counts in sorted(node_count_match_p.items())}
matches_fittable_p = np.array(list(sorted_node_counts_p.keys()), dtype=float) # converts to fittable form
node_counts_fittable_p = np.array(list(sorted_node_counts_p.values()), dtype=float) 

sorted_leaves_counts_p = {matches: counts for matches, counts in sorted(leaves_count_match_p.items())}
matches_leaves_fittable_p = np.array(list(sorted_leaves_counts_p.keys()), dtype=float) # converts to fittable form
leaves_counts_fittable_p = np.array(list(sorted_leaves_counts_p.values()), dtype=float) 

Fit the power-law model — first flat pass.

In [ ]:
bestfit_node_counts, bestfit_cov_node_counts = opt.curve_fit(count_match_func, matches_fittable, node_counts_fittable, p0=initial_guesses, maxfev=100000)
bestfit_leaves_counts, bestfit_cov_leaves_counts = opt.curve_fit(count_match_func, matches_leaves_fittable, leaves_counts_fittable, p0=initial_guesses, maxfev=100000)

Fit the power-law model — second flat pass.

In [ ]:
bestfit_node_counts_p, bestfit_cov_node_counts_p = opt.curve_fit(count_match_func, matches_fittable_p, node_counts_fittable_p, p0=initial_guesses, maxfev=100000)
bestfit_leaves_counts_p, bestfit_cov_leaves_counts_p = opt.curve_fit(count_match_func, matches_leaves_fittable_p, leaves_counts_fittable_p, p0=initial_guesses, maxfev=100000)

Plot node/leaf counts vs. year and vs. cumulative matches, with fitted curves — first flat pass.

In [ ]:
# counts by year
plt.figure(figsize=(5,5))
plt.scatter(node_count_dict.keys(), node_count_dict.values(), color='steelblue')
plt.scatter(leaves_count_dict.keys(), leaves_count_dict.values(), color='palevioletred')
plt.ylabel('Count', size=16)
plt.xlabel('Year', size=16)
plt.yscale('log')


# counts by matches
plt.figure(figsize=(5,5))
plt.scatter(node_count_match.keys(), node_count_match.values(), color='steelblue')
plt.plot(matches_fittable, count_match_func(matches_fittable, bestfit_node_counts[0], bestfit_node_counts[1]), color='steelblue')
plt.scatter(leaves_count_match.keys(), leaves_count_match.values(), color='palevioletred')
plt.plot(matches_leaves_fittable, count_match_func(matches_leaves_fittable, bestfit_leaves_counts[0], bestfit_leaves_counts[1]), color='palevioletred')
plt.yscale('log')
plt.xscale('log')
plt.ylabel('Count', size=16)
plt.xlabel('Cumulative matches', size=16)
plt.annotate('NODES -- slope: ' + str(round(bestfit_node_counts[1],3)), (7, 1000), color='steelblue', size=12)
plt.annotate('LEAVES -- slope: ' + str(round(bestfit_leaves_counts[1],3)), (7, 800), color='palevioletred', size=12)

Same plot for the second flat pass.

In [ ]:
# counts by year
plt.figure(figsize=(5,5))
plt.scatter(node_count_dict_p.keys(), node_count_dict_p.values(), color='teal')
plt.scatter(leaves_count_dict_p.keys(), leaves_count_dict_p.values(), color='darkblue')
plt.ylabel('Count', size=16)
plt.xlabel('Year', size=16)
plt.yscale('log')


# counts by matches
plt.figure(figsize=(5,5))
plt.scatter(node_count_match_p.keys(), node_count_match_p.values(), color='teal', label='NODES -- slope: ' + str(round(bestfit_node_counts_p[1],3)))
plt.plot(matches_fittable_p, count_match_func(matches_fittable_p, bestfit_node_counts_p[0], bestfit_node_counts_p[1]), color='teal')
plt.scatter(leaves_count_match_p.keys(), leaves_count_match_p.values(), color='darkblue', label='LEAVES -- slope: ' + str(round(bestfit_leaves_counts_p[1],3)))
plt.plot(matches_leaves_fittable_p, count_match_func(matches_leaves_fittable_p, bestfit_leaves_counts_p[0], bestfit_leaves_counts_p[1]), color='darkblue')
plt.yscale('log')
plt.xscale('log')
plt.ylabel('Count', size=16)
plt.xlabel('Cumulative matches', size=16)
plt.legend()
#plt.annotate('NODES -- slope: ' + str(round(bestfit_node_counts_p[1],3)), (7, 1000), color='teal', size=12)
#plt.annotate('LEAVES -- slope: ' + str(round(bestfit_leaves_counts_p[1],3)), (7, 800), color='darkblue', size=12)

Two ways of expressing the growth rate implied by the fitted node-count exponent (second flat pass).

In [ ]:
alpha_growth2=2**(1/bestfit_node_counts_p[1])
print(alpha_growth2)

## Node/leaf counts vs. rulebook word count

Reloads the raw rulebook texts (same approach as the text-visualization
notebook) so node/leaf counts can be related to word count rather than
match count.

In [ ]:
folder2003_path = (TEXTS_DIR / '2003-2016')
folder2018_path = (TEXTS_DIR / '2018-2026')
folder2000_path = (TEXTS_DIR / '2000-2002')
folder1997_path = (TEXTS_DIR / '1997-1998')
folder1985_path = (TEXTS_DIR / '1985-1995')
folder1969_path = (TEXTS_DIR / '1969-1974')
folder1954_path = (TEXTS_DIR / '1954-1959')
folder1934_path = (TEXTS_DIR / '1934-1949')
folder1929_path = (TEXTS_DIR / '1929')
folder1894_path = (TEXTS_DIR / '1894-1924')
folder1874_path = (TEXTS_DIR / '1874-1889')
folder1845_path = (TEXTS_DIR / '1845')


rulebooks = {} # empty dictionary to save data

# cycle through all files
for file_path in (folder2003_path.glob('*.txt')):
    with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
        file_name = file_path.stem

        # same dictionary key as integer of just year
        year = file_name.replace('clean_', '')
        rulebooks[int(year)] = file.read()
for file_path in (folder2018_path.glob('*.txt')):
    with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
        file_name = file_path.stem

        # same dictionary key as integer of just year
        year = file_name.replace('clean_', '')
        rulebooks[int(year)] = file.read()
for file_path in (folder2000_path.glob('*.txt')):
    with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
        file_name = file_path.stem

        # same dictionary key as integer of just year
        year = file_name.replace('clean_', '')
        rulebooks[int(year)] = file.read()
for file_path in (folder1997_path.glob('*.txt')):
    with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
        file_name = file_path.stem

        # same dictionary key as integer of just year
        year = file_name.replace('clean_', '')
        rulebooks[int(year)] = file.read()
for file_path in (folder1985_path.glob('*.txt')):
    with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
        file_name = file_path.stem

        # same dictionary key as integer of just year
        year = file_name.replace('clean_', '')
        rulebooks[int(year)] = file.read()
for file_path in (folder1969_path.glob('*.txt')):
    with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
        file_name = file_path.stem

        # same dictionary key as integer of just year
        year = file_name.replace('clean_', '')
        rulebooks[int(year)] = file.read()
for file_path in (folder1954_path.glob('*.txt')):
    with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
        file_name = file_path.stem

        # same dictionary key as integer of just year
        year = file_name.replace('clean_', '')
        rulebooks[int(year)] = file.read()
for file_path in (folder1934_path.glob('*.txt')):
    with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
        file_name = file_path.stem

        # same dictionary key as integer of just year
        year = file_name.replace('clean_', '')
        rulebooks[int(year)] = file.read()
for file_path in (folder1929_path.glob('*.txt')):
    with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
        file_name = file_path.stem

        # same dictionary key as integer of just year
        year = file_name.replace('clean_', '')
        rulebooks[int(year)] = file.read()
for file_path in (folder1894_path.glob('*.txt')):
    with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
        file_name = file_path.stem

        # same dictionary key as integer of just year
        year = file_name.replace('clean_', '')
        rulebooks[int(year)] = file.read()
for file_path in (folder1874_path.glob('*.txt')):
    with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
        file_name = file_path.stem

        # same dictionary key as integer of just year
        year = file_name.replace('clean_', '')
        rulebooks[int(year)] = file.read()
for file_path in (folder1845_path.glob('*.txt')):
    with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
        file_name = file_path.stem

        # same dictionary key as integer of just year
        year = file_name.replace('clean_', '')
        rulebooks[int(year)] = file.read()

Tokenize each rulebook and compute its word count.

In [ ]:
rulebooks_token = {}
rulebooks_lengths = {}
rulebooks_uniq_lengths = {}

# creating dictionary year: tokenized words
# creates dictionaries with year: # of (unique) words
for book in rulebooks:
    rulebooks_token[book] = nltk.word_tokenize(rulebooks[book].lower())
    rulebooks_token[book] = [word for word in rulebooks_token[book] if word.isalnum()] # removes characters
    rulebooks_lengths[book] = len(rulebooks_token[book])
    rulebooks_uniq_lengths[book] = len(set(rulebooks_token[book]))

Build a `word count -> node/leaf count` mapping (second flat pass).

In [ ]:
nodes_length_p = {}
leaves_length_p = {}

# make dictionary word count: nodes or leaves
for year in rulebooks_lengths: 
    length_temp = rulebooks_lengths[year]

    nodes_length_p[length_temp] = node_count_dict_p[year]
    leaves_length_p[length_temp] = leaves_count_dict_p[year]


Sort and convert to fittable arrays.

In [ ]:
sorted_node_length_counts_p = {length: counts for length, counts in sorted(nodes_length_p.items())}
length_fittable_p = np.array(list(sorted_node_length_counts_p.keys()), dtype=float) # converts to fittable form
node_counts_length_fittable_p = np.array(list(sorted_node_length_counts_p.values()), dtype=float) 

sorted_leaves_length_counts_p = {length: counts for length, counts in sorted(leaves_length_p.items())}
length_leaves_fittable_p = np.array(list(sorted_leaves_length_counts_p.keys()), dtype=float) # converts to fittable form
leaves_counts_length_fittable_p = np.array(list(sorted_leaves_length_counts_p.values()), dtype=float) 

Fit the power-law model.

In [ ]:
bestfit_node_counts_length_p, bestfit_cov_node_length_counts_p = opt.curve_fit(count_match_func, length_fittable_p, node_counts_length_fittable_p, p0=initial_guesses, maxfev=100000)
bestfit_leaves_counts_length_p, bestfit_cov_leaves_length_counts_p = opt.curve_fit(count_match_func, length_leaves_fittable_p, leaves_counts_length_fittable_p, p0=initial_guesses, maxfev=100000)

Plot node/leaf counts vs. word count, with fitted curves.

In [ ]:
# counts by matches
plt.figure(figsize=(5,5))
plt.scatter(nodes_length_p.keys(), nodes_length_p.values(), color='steelblue')
plt.plot(length_fittable_p, count_match_func(length_fittable_p, bestfit_node_counts_length_p[0], bestfit_node_counts_length_p[1]), color='steelblue')
plt.scatter(leaves_length_p.keys(), leaves_length_p.values(), color='palevioletred')
plt.plot(length_leaves_fittable_p, count_match_func(length_leaves_fittable_p, bestfit_leaves_counts_length_p[0], bestfit_leaves_counts_length_p[1]), color='palevioletred')
plt.yscale('log')
plt.xscale('log')
plt.ylabel('# of nodes', size=16)
plt.xlabel('# of words', size=16)
plt.annotate('NODES -- slope: ' + str(round(bestfit_node_counts_length_p[1],3)), (1000, 1000), color='steelblue', size=12)
plt.annotate('LEAVES -- slope: ' + str(round(bestfit_leaves_counts_length_p[1],3)), (1000, 800), color='palevioletred', size=12)